# Simple RAG (Retrieval-Augmented Generation) System for CSV Files with Google Gemini

## Overview

This notebook implements a basic Retrieval-Augmented Generation (RAG) system for processing and querying CSV documents using **Google Gemini** direct APIs.

## Key Components

1. **Environment Setup**: Loading credentials from a `.env` file.
2. **Data Loading**: Using `pandas` for efficient CSV processing.
3. **Vectorization**: Using Google Gemini's `models/gemini-embedding-001` for text embeddings.
4. **Vector Store**: Using **FAISS** directly for similarity search.
5. **Synthesis**: Using Gemini (`gemini-3-flash-preview`) for generating final answers.




![overall](https://raw.githubusercontent.com/shubhra-opensource/rag-systems-cookbook/refs/heads/main/01_Jupyter_Notebooks/images/02.png)

### 2. Imports and Configuration

We load the API key from the `.env` file and configure the Google Generative AI library.

In [ ]:
import os
import pandas as pd
import numpy as np
import faiss
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Configure Gemini
gemini_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("Please set GOOGLE_API_KEY or GEMINI_API_KEY in your .env file")

genai.configure(api_key=gemini_api_key)

# Model Definitions
EMBEDDING_MODEL = "models/gemini-embedding-001"
GENFRATION_MODEL = "gemini-3-flash-preview"

### 3. Data Loading

We download the sample CSV data if it doesn't exist and load it using `pandas`.

In [ ]:
file_path = 'data/customers-100.csv'
df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")
df = df.drop(columns=['Index'])
df.head()

### 4. Vector Store Creation

We convert each row into a text string, generate embeddings using Gemini, and store them in a FAISS index.

In [ ]:
def format_row(row):
    return " ".join([f"{col} is {val};" for col, val in row.items()])

# Convert rows to strings
documents = df.apply(format_row, axis=1).tolist()

In [ ]:
# documents

In [ ]:
print("Generating embeddings (this may take a moment)...")

# Function to get embeddings in batches
def get_embeddings(texts, model=EMBEDDING_MODEL):
    # Gemini allows batch embedding
    result = genai.embed_content(
        model=model,
        content=texts,
        task_type="retrieval_document"
    )
    return np.array(result['embedding'])

In [ ]:
# Get all embeddings
embeddings = get_embeddings(documents)

In [ ]:
# Initialize FAISS index
dimension = embeddings.shape[1]
dimension

In [ ]:
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings.astype('float32'))

print(f"FAISS index built with {faiss_index.ntotal} vectors.")

### 5. Retrieval and Question Answering

We define a function that takes a query, retrieves relevant context from FAISS, and uses Gemini to synthesize the final answer.

In [ ]:
def ask_gemini(query, top_k=3):
    # 1. Embed the query
    query_embedding = genai.embed_content(
        model=EMBEDDING_MODEL,
        content=query,
        task_type="retrieval_query"
    )['embedding']
    
    # 2. Search FAISS
    distances, indices = faiss_index.search(np.array([query_embedding]).astype('float32'), top_k)
    
    # 3. Prepare context
    retrieved_docs = [documents[i] for i in indices[0]]
    context = "\n---\n".join(retrieved_docs)
    
    # 4. Generate answer
    prompt = f"""
    You are an assistant for question-answering tasks. 
    Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, say that you don't know. 
    Use three sentences maximum and keep the answer concise.

    Context:
    {context}

    Question: {query}

    Answer:
    """
    
    model = genai.GenerativeModel(GENFRATION_MODEL)
    response = model.generate_content(prompt)
    
    return response.text

# Test the RAG system
question = "Which company does Sheryl Baxter work for?"
answer = ask_gemini(question)
print(f"Question: {question}")
print(f"Answer: {answer}")

### 6. Additional Queries

Testing with more questions derived from the data.

In [ ]:
queries = [
    "What is the email of Preston Lozano?",
    "Which country is Linda Olsen from?",
    "What is the website for Rasmussen Group?",
    "Which people having hogan in their email id?"
]

for q in queries:
    print(f"Q: {q}")
    print(f"A: {ask_gemini(q)}\n")